# LlamaParse document extractor

Point `INPUT_PATH` at a **PDF or Excel** file, run all cells, and get:

| | |
|---|---|
| **What LlamaParse read** | the parsed markdown, its character count, and a copy on disk |
| **What the pipeline found** | technical facts, deviations and commercial terms extracted from that markdown |

Design: [`2026-08-12-llamaparse-extraction-notebook-design.md`](../docs/superpowers/specs/2026-08-12-llamaparse-extraction-notebook-design.md)

LlamaParse is the **only** reader here — for spreadsheets as well as PDFs, which
is not what the shipped pipeline does and takes the notebook-local patch in
step 3. `.docx` and `.txt` in a folder or `.zip` still go through the shipped
readers.

This is an instrument for looking at one document. It is **not** a re-run of the
parser A/B, which compared LlamaParse against `pdftotext` on a controlled corpus
and [recorded a negative result](../docs/pdf-parser-quality-ab-results.md).

Two things cost money: the LlamaParse parse in step 5 (per page, by tier, and
only on a cache miss) and the Anthropic extraction in step 6 (gated on
`RUN_EXTRACT`).


In [ ]:
# ── 1. CONFIG ─────────────────────────────────────────────────────────────

# The document to read: a .pdf, a .xlsx, a folder of them, or a vendor .zip.
#
# The default is MKON's spreadsheet — the parser A/B's control document, the
# one file in that corpus whose reader the A/B could not vary, because
# read_text_with_source hard-routes .xlsx to openpyxl in every arm. It is
# therefore the document this notebook's patch exists to read differently.
INPUT_PATH = r"processed-data/02-vendor-bids/MKON/Copy of ADN-AEC-ME-SPC-026_MR_Gas_Genset.xlsx"

# LlamaParse tier: fast | cost_effective | agentic | agentic_plus.
# `agentic` is the accurate per-page tier and what llamaparse_reader defaults
# to. The parse cache is keyed by (content hash, tier), so changing this
# re-parses — and re-bills — every file.
TIER = "agentic"

# True  -> run the real extractors against Anthropic after parsing.
# False -> parse only. LlamaParse still bills; Anthropic does not.
RUN_EXTRACT = True

MODEL = "claude-sonnet-5"

# Where the parsed markdown and facts.csv land. None -> notebooks/parsed/
OUTPUT_DIR = None

PREVIEW_CHARS = 1500  # of parsed markdown to show per document

# None -> the containing folder's name, which is the vendor in this corpus's
# layout (processed-data/02-vendor-bids/<VENDOR>/<document>). Not the file
# stem: that labels the vendor with the document's own name and repeats it in
# every output filename.
VENDOR_NAME  = None
SCRATCH_ROOT = None   # None -> a fresh temp dir. Do not point this at projects/.


In [ ]:
# ── 2. SETUP ──────────────────────────────────────────────────────────────
import contextlib
import logging
import os
import shutil
import sys
import tempfile
from pathlib import Path

logging.getLogger("pypdf").setLevel(logging.ERROR)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "procurement").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("run this notebook from inside the repo")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

try:
    import dotenv
    dotenv.load_dotenv(REPO_ROOT / ".env")
except ImportError:
    pass

from procurement import chunking, llamaparse_reader, loaders
from procurement import pipeline as pipeline_mod
from procurement import project as proj
from procurement.extract_requirements import REQUIREMENTS_CHUNK_CHARS
from procurement.extract_tech import TECH_CHUNK_CHARS
from procurement.pipeline import run_ingestion
from procurement.store import snapshots

# llamaparse_reader resolves its cache directory relative to the *working*
# directory. A notebook run from notebooks/ would therefore miss a cache
# written from the repo root and re-bill every file. Pin it.
os.environ[llamaparse_reader.CACHE_DIR_ENV] = str(
    REPO_ROOT / llamaparse_reader.DEFAULT_CACHE_DIR)

OUTPUT_DIR = Path(OUTPUT_DIR) if OUTPUT_DIR else REPO_ROOT / "notebooks" / "parsed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

print(f"repo         {REPO_ROOT}")
print(f"tier         {TIER}")
print(f"parse cache  {os.environ[llamaparse_reader.CACHE_DIR_ENV]}")
print(f"output       {OUTPUT_DIR}")
print(f"chunk budget tech={TECH_CHUNK_CHARS:,}  requirements={REQUIREMENTS_CHUNK_CHARS:,}")
if not os.getenv(llamaparse_reader.API_KEY_ENV):
    print(f"\n!! {llamaparse_reader.API_KEY_ENV} is not set — only files already "
          "in the parse cache can be read.")


In [ ]:
# ── 3. THE READER PATCH — LlamaParse for spreadsheets too ─────────────────
# `loaders.read_text_with_source` consults `pdf_reader` only for .pdf; .xlsx is
# hard-routed to openpyxl in every arm, which is exactly what made the .xlsx
# document a control in the parser A/B. So Excel-through-LlamaParse cannot come
# from the shipped switch.
#
# Patching `loaders.read_xlsx_text` would work, but the "xlsx" label in
# read_text_with_source is a literal, so the store would record `text_source =
# "xlsx"` for text a hosted parser produced. text_source is the one field that
# tells "this datasheet states nothing" apart from "we read it with the wrong
# reader"; a run that lies there has destroyed the only thing it is for.
#
# So patch pipeline.py's two module-scope bindings instead (pipeline.py:26-27),
# which lets the wrapper name its own reader. Nothing in procurement/ or
# shared/ is modified; the context manager removes both on exit.
LLAMAPARSE_EXTS = {".pdf", ".xlsx", ".xls"}


def llamaparse_read(path, llm_fallback=None, pdf_reader=None):
    """(text, reader) with LlamaParse for PDFs and spreadsheets.

    `parse_pdf` is a misnomer: it uploads bytes and asks LlamaCloud for
    markdown, and nothing in it is PDF-specific. Calling it for .xlsx here is
    preferable to adding a `parse_any` alias to shipped code for one notebook.

    `llm_fallback` is deliberately not forwarded on the LlamaParse arm. If the
    parser fails it raises, `_read_and_guard` records the document `failed`
    with the reason, and that is the honest outcome — an LLM transcription
    rescue would report another reader's text under LlamaParse's name.
    """
    if Path(path).suffix.lower() in LLAMAPARSE_EXTS:
        return llamaparse_reader.parse_pdf(path, tier=TIER), "llamaparse"
    return loaders.read_text_with_source(path, llm_fallback=llm_fallback,
                                         pdf_reader=pdf_reader)


def llamaparse_read_text(path, llm_fallback=None, pdf_reader=None):
    return llamaparse_read(path, llm_fallback=llm_fallback,
                           pdf_reader=pdf_reader)[0]


@contextlib.contextmanager
def llamaparse_everywhere():
    """Both bindings, not one. `_classify_pass` uses `read_text` for the
    500-char head it shows the classifier; leaving it shipped would classify a
    spreadsheet on openpyxl's rendering and extract it from LlamaParse's — two
    readings of one document in a single run. It costs nothing extra: the parse
    is cached on disk by the pass below."""
    originals = (pipeline_mod.read_text_with_source, pipeline_mod.read_text)
    pipeline_mod.read_text_with_source = llamaparse_read
    pipeline_mod.read_text = llamaparse_read_text
    try:
        yield
    finally:
        pipeline_mod.read_text_with_source, pipeline_mod.read_text = originals


# ── the Anthropic usage recorder ──────────────────────────────────────────
# `classify_structure` reads `message.usage` and throws it away (pricing spec
# §5.4). Record one layer lower: anthropic_client.py constructs
# `anthropic.Anthropic(...)` inside the call, so a swapped factory sees every
# request.
import anthropic

_REAL_ANTHROPIC = anthropic.Anthropic


class _RecordingMessages:
    def __init__(self, inner, sink):
        self._inner = inner
        self._sink = sink

    def create(self, **kwargs):
        message = self._inner.create(**kwargs)
        usage = getattr(message, "usage", None)
        self._sink.append({
            "model": kwargs.get("model"),
            "input_tokens": getattr(usage, "input_tokens", 0) or 0,
            "output_tokens": getattr(usage, "output_tokens", 0) or 0,
            "cache_write": getattr(usage, "cache_creation_input_tokens", 0) or 0,
            "cache_read": getattr(usage, "cache_read_input_tokens", 0) or 0,
            "stop_reason": getattr(message, "stop_reason", None),
        })
        return message

    def __getattr__(self, name):
        return getattr(self._inner, name)


class _RecordingClient:
    def __init__(self, inner, sink):
        self._inner = inner
        self.messages = _RecordingMessages(inner.messages, sink)

    def __getattr__(self, name):
        return getattr(self._inner, name)


@contextlib.contextmanager
def record_usage():
    sink = []
    anthropic.Anthropic = lambda *a, **kw: _RecordingClient(_REAL_ANTHROPIC(*a, **kw), sink)
    try:
        yield sink
    finally:
        anthropic.Anthropic = _REAL_ANTHROPIC


# Cache multipliers, named rather than inlined so they stay legibly the same
# figures as document_cost_estimator.ipynb. Algebraically identical to the
# literals they replace — this changes no number, only where the number lives.
CACHE_READ_MULTIPLIER  = 0.10   # cache hits bill at 0.1x the input rate
CACHE_WRITE_MULTIPLIER = 1.25   # cache writes at 1.25x

# Sonnet 5 rate card, USD per MTok. Two rows: the introductory rate expires
# 2026-08-31 and a figure quoted today has to survive that date.
RATE_CARD = {
    "Sonnet 5 · intro (to 2026-08-31)": {"in": 2.00, "out": 10.00},
    "Sonnet 5 · standard":              {"in": 3.00, "out": 15.00},
}


def cost_usd(rate, input_tokens, output_tokens, cache_read=0, cache_write=0):
    return (
        input_tokens * rate["in"]
        + cache_write * rate["in"] * CACHE_WRITE_MULTIPLIER
        + cache_read * rate["in"] * CACHE_READ_MULTIPLIER
        + output_tokens * rate["out"]
    ) / 1_000_000

print("patch ready — not yet installed; step 6 installs it around the run")


In [ ]:
# ── 4. STAGE THE INPUT ────────────────────────────────────────────────────
# One throwaway project under the temp directory. Everything lands on the
# vendor side: without an RFQ there is no compliance matrix, but classify,
# route and extract all run exactly as they do in production.
source = Path(INPUT_PATH)
if not source.is_absolute():
    source = (REPO_ROOT / source).resolve()
if not source.exists():
    raise FileNotFoundError(f"INPUT_PATH does not exist: {source}")

ROOT = Path(SCRATCH_ROOT) if SCRATCH_ROOT else Path(tempfile.mkdtemp(prefix="llamaparse-"))
if ROOT.resolve() == (REPO_ROOT / "projects").resolve():
    raise ValueError(
        "SCRATCH_ROOT must not be projects/ — test_real_corpus_coverage.py "
        "asserts against the newest multi-vendor store there.")
ROOT.mkdir(parents=True, exist_ok=True)

project = proj.create_project(str(ROOT), f"llamaparse {source.stem}")
SLUG = project.slug
PROJECT_DIR = ROOT / SLUG

if source.suffix.lower() == ".zip":
    VENDORS = proj.unpack_vendor_zip(str(ROOT), SLUG, str(source))
else:
    # A folder is its own vendor; a single file belongs to the vendor whose
    # folder holds it. Using the file's stem instead would name the vendor
    # after the document and repeat that name in every artefact below.
    derived = source.name if source.is_dir() else source.parent.name
    vendor = proj.slugify(VENDOR_NAME or derived) or "vendor"
    dest = PROJECT_DIR / "vendors" / vendor
    dest.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(source, dest / source.name)
    project.vendors = proj.list_vendor_dirs(str(ROOT), SLUG)
    proj.save_project(str(ROOT), project)
    VENDORS = project.vendors

# `vendor_files` drops Excel's `~$` lock files and dotfiles, so a folder staged
# straight from a working corpus does not ingest a 165-byte lock file as a
# document — and does not pay LlamaParse to parse one.
STAGED = [(v, f) for v in VENDORS for f in proj.vendor_files(str(ROOT), SLUG, v)]
print(f"input    {source}")
print(f"staged   {len(STAGED)} file(s) across {len(VENDORS)} vendor(s): {', '.join(VENDORS)}")
print(f"project  {PROJECT_DIR}")


In [ ]:
# ── 5. PARSE PASS — LlamaParse reads every file. This bills. ──────────────
# A cache miss is a billable parse; a hit costs nothing. The cache is keyed on
# (content hash, tier), so re-running this cell is free until TIER changes.
rows = []
texts = []

for vendor, path in STAGED:
    f = Path(path)
    size = f.stat().st_size
    ext = f.suffix.lower()
    billable = (ext in LLAMAPARSE_EXTS
                and not Path(llamaparse_reader.cache_path(path, TIER)).exists())
    try:
        text, reader = llamaparse_read(path)
        note = ""
    except Exception as exc:
        text, reader = "", "failed"
        note = f"{type(exc).__name__}: {exc}"[:120]

    texts.append(text)
    if text:
        out = OUTPUT_DIR / f"{vendor}__{f.stem}.md"
        out.write_text(text, encoding="utf-8")

    rows.append({
        "rel": f.relative_to(PROJECT_DIR).as_posix(),
        "vendor": vendor,
        "file": f.name,
        "reader": reader,
        "MB": round(size / 1_048_576, 3),
        "chars": len(text),
        "billable": billable,
        "extractable": len(text.strip()) >= loaders.MIN_EXTRACTABLE_CHARS,
        "tech_chunks": len(chunking.chunk_on_lines(text, TECH_CHUNK_CHARS)),
        "req_chunks": len(chunking.chunk_on_lines(text, REQUIREMENTS_CHUNK_CHARS)),
        "note": note,
    })

parsed = pd.DataFrame(rows)
display(parsed.drop(columns=["rel"]))

print(f"\n{len(parsed)} file(s), {int(parsed['chars'].sum()):,} chars parsed")
print(f"{int(parsed['billable'].sum())} billable parse(s) at tier {TIER!r}; "
      f"the rest came from the cache")
print(f"not extractable: {int((~parsed['extractable']).sum())} "
      f"(under {loaders.MIN_EXTRACTABLE_CHARS} chars — recorded `failed`, no extractor runs)")
print(f"markdown written to {OUTPUT_DIR}")
print("\nLlamaParse bills per page by tier. `parse_pdf` discards the page count "
      "before\nany caller sees it, so this reports whether a parse was billable, "
      "not what it cost.\nCredits are on the LlamaCloud dashboard.")

for row, text in zip(rows, texts):
    print(f"\n{'─' * 76}\n{row['file']}  ({row['chars']:,} chars via {row['reader']})\n{'─' * 76}")
    print(text[:PREVIEW_CHARS] or row["note"] or "(empty)")


---
## Everything above is LlamaParse. The next cell spends Anthropic tokens.

It runs `run_ingestion` — classify → lineage → route → extract — with the patch
from step 3 installed, so every extractor reads the LlamaParse markdown you just
previewed, and records the token count of every request.

Set `RUN_EXTRACT = False` in step 1 to stop here and keep this a parse-only
instrument.


In [ ]:
# ── 6. EXTRACT PASS — the real pipeline over the LlamaParse text. ─────────
usage = []
result = None

if not RUN_EXTRACT:
    print("RUN_EXTRACT is False — nothing extracted, no Anthropic spend.")
elif not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY is not set — put it in .env at the repo root.")
else:
    from shared.llm.anthropic_client import AnthropicClient

    client = AnthropicClient(model=MODEL)
    # No pdf_fallback: the LLM transcription rescue must not stand in for a
    # LlamaParse failure, which belongs in the record as a failure.
    with llamaparse_everywhere(), record_usage() as usage:
        result = run_ingestion(str(ROOT), SLUG, client, pdf_reader="llamaparse")
    print(f"run finished: {result}")
    print(f"{len(usage)} Anthropic request(s) on {MODEL}")


In [ ]:
# ── 7. WHAT THE PIPELINE FOUND ───────────────────────────────────────────
documents = snapshots.load_documents(str(ROOT), SLUG) if result else []

if not documents:
    print("nothing ingested — RUN_EXTRACT was False.")
else:
    docs = pd.DataFrame([{
        "vendor": d.vendor,
        "file": Path(d.path).name,
        "class": d.doc_class,
        "text_source": d.text_source or "—",
        "status": d.extraction_status,
        "note": (d.notes or "")[:70],
    } for d in documents])
    print("DOCUMENTS")
    display(docs)

fact_rows, deviation_rows = [], []
for vendor in VENDORS:
    facts = snapshots.load_facts(str(ROOT), SLUG, vendor) if result else None
    if facts is None:
        continue
    for fact in facts.technical:
        fact_rows.append({"vendor": vendor, **{k: fact.get(k) for k in
                          ("parameter", "value", "unit", "verbatim", "doc_id")}})
    for dev in facts.deviations:
        deviation_rows.append({"vendor": vendor, **{k: dev.get(k) for k in
                               ("clause_ref", "statement", "disposition", "doc_id")}})
    if facts.commercial:
        print(f"\nCOMMERCIAL — {vendor}")
        display(pd.Series(facts.commercial))
    if facts.normalized:
        print(f"\nNORMALIZED — {vendor}")
        display(pd.Series(facts.normalized))

if fact_rows:
    tech = pd.DataFrame(fact_rows)
    print(f"\nTECHNICAL FACTS — {len(tech)}")
    display(tech)
    tech.to_csv(OUTPUT_DIR / "facts.csv", index=False)
    print(f"written to {OUTPUT_DIR / 'facts.csv'}")
elif documents:
    # Absent, not zero: no parameter was found, and saying so is the finding.
    print("\nTECHNICAL FACTS — none. Check `status` and `note` above for why.")

if deviation_rows:
    print(f"\nDEVIATIONS — {len(deviation_rows)}")
    display(pd.DataFrame(deviation_rows))

# ── Anthropic cost ───────────────────────────────────────────────────────
if usage:
    calls = pd.DataFrame(usage)
    tok_in, tok_out = int(calls["input_tokens"].sum()), int(calls["output_tokens"].sum())
    cache_read, cache_write = int(calls["cache_read"].sum()), int(calls["cache_write"].sum())
    truncated = calls[calls["stop_reason"] == "max_tokens"]
    if len(truncated):
        print(f"\n!! {len(truncated)} response(s) hit the max_tokens ceiling — "
              "the extraction they belong to failed. See the notes above.")
    print(f"\nANTHROPIC — measured across {len(calls)} request(s)")
    print(f"  input {tok_in:,}   output {tok_out:,}   "
          f"cache read {cache_read:,}   cache write {cache_write:,}")
    display(pd.DataFrame([
        {"rate card": name,
         "USD": round(cost_usd(rate, tok_in, tok_out, cache_read, cache_write), 4)}
        for name, rate in RATE_CARD.items()
    ]))


In [ ]:
# ── 8. SELF-CHECK — did the patch actually take? ─────────────────────────
# Without this the notebook's worst failure is silent: openpyxl's rendering of
# a spreadsheet, presented above as LlamaParse's. `text_source` is recorded as
# "<reader>:<n>chars", so the reader name is the prefix.
SHIPPED_READERS = ("xlsx:", "pdftotext:", "pypdf:", "docx:", "text:", "llm:")

wrong, unread = [], []
for d in documents:
    if Path(d.path).suffix.lower() not in LLAMAPARSE_EXTS:
        continue
    source_name = d.text_source or ""
    if source_name.startswith(SHIPPED_READERS):
        wrong.append((Path(d.path).name, source_name))
    elif not source_name.startswith("llamaparse:"):
        unread.append((Path(d.path).name, source_name, (d.notes or "")[:80]))

if wrong:
    raise AssertionError(
        "the LlamaParse patch did NOT take for: "
        + "; ".join(f"{n} read by {s}" for n, s in wrong)
        + " — every number above for those files is another reader's.")
if unread:
    print("LlamaParse failed on (recorded `failed`, which is the honest outcome):")
    for name, src, note in unread:
        print(f"  {name}  [{src}]  {note}")
if documents and not wrong and not unread:
    print(f"OK — all {len([d for d in documents if Path(d.path).suffix.lower() in LLAMAPARSE_EXTS])} "
          "PDF/Excel document(s) were read by LlamaParse.")
elif not documents:
    print("nothing to check — RUN_EXTRACT was False.")


In [ ]:
# ── 9. CLEAN UP ──────────────────────────────────────────────────────────
# Drops the scratch project. Keeps the parse cache (credits already spent) and
# OUTPUT_DIR (the markdown and facts.csv).
#
# Re-staging from scratch is also how you get a clean second measurement:
# run_ingestion caches on content_sha256, so a second run of the same bytes
# asks Anthropic almost nothing.
if SCRATCH_ROOT is None:
    shutil.rmtree(ROOT, ignore_errors=True)
    print(f"removed {ROOT}")
else:
    print(f"kept {ROOT} (SCRATCH_ROOT was set explicitly)")
